In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Deterministic seed
np.random.seed(42)
random.seed(42)

# ===== ABSOLUTE BASE PATH (FIXED) =====
BASE_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair"

# Paths (DO NOT CHANGE VARIABLE NAMES)
CLEAN_PATH = os.path.join(BASE_PATH, "data/cleaned/")
SYNTHETIC_PATH = os.path.join(BASE_PATH, "data/synthetic/")
REPORT_PATH = os.path.join(BASE_PATH, "reports/")

# Ensure directories exist
os.makedirs(CLEAN_PATH, exist_ok=True)
os.makedirs(SYNTHETIC_PATH, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)

In [69]:
# Generate clean dataset
n = 1000

df_clean = pd.DataFrame({
    "id": range(1, n + 1),
    "name": [f"name_{i}" for i in range(1, n + 1)],
    "age": np.random.randint(18, 60, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "salary": np.random.randint(20000, 100000, n),
    "join_date": [
        datetime(2015, 1, 1) + timedelta(days=int(x))
        for x in np.random.randint(0, 2000, n)
    ],
    "email": [f"user{i}@example.com" for i in range(1, n + 1)]
})

# Save clean dataset
df_clean.to_csv(os.path.join(CLEAN_PATH, "clean_dataset.csv"), index=False)

In [70]:
df_original = pd.read_csv(os.path.join(CLEAN_PATH, "clean_dataset.csv"))
df_working = df_original.copy()

In [71]:
corruption_log = []

In [72]:
for col in ["age", "salary", "gender"]:
    idx = np.random.choice(df_working.index, size=int(0.1 * len(df_working)), replace=False)
    df_working.loc[idx, col] = np.nan
    
    for i in idx:
        corruption_log.append({
            "row": int(i),
            "column": col,
            "issue": "missing_value"
        })

In [73]:
duplicate_rows = df_working.sample(frac=0.05, random_state=42)
df_working = pd.concat([df_working, duplicate_rows], ignore_index=True)

for i in duplicate_rows.index:
    corruption_log.append({
        "row": int(i),
        "column": "all",
        "issue": "duplicate_row"
    })

In [74]:
mapping = {"Male": "M", "Female": "F"}

idx = np.random.choice(df_working.index, size=int(0.2 * len(df_working)), replace=False)

for i in idx:
    val = df_working.loc[i, "gender"]
    if val in mapping:
        df_working.loc[i, "gender"] = mapping[val]
        corruption_log.append({
            "row": int(i),
            "column": "gender",
            "issue": "categorical_inconsistency"
        })

In [75]:
idx = np.random.choice(df_working.index, size=int(0.2 * len(df_working)), replace=False)

for i in idx:
    df_working.loc[i, "name"] = " " + str(df_working.loc[i, "name"]) + " "
    
    corruption_log.append({
        "row": int(i),
        "column": "name",
        "issue": "whitespace_noise"
    })

In [76]:
idx = np.random.choice(df_working.index, size=int(0.15 * len(df_working)), replace=False)

for i in idx:
    date_val = df_working.loc[i, "join_date"]
    
    try:
        dt = pd.to_datetime(date_val)
        df_working.loc[i, "join_date"] = dt.strftime("%d-%m-%Y")  # inconsistent format
        
        corruption_log.append({
            "row": int(i),
            "column": "join_date",
            "issue": "date_format_error"
        })
    except:
        pass

In [77]:
idx = np.random.choice(df_working.index, size=int(0.05 * len(df_working)), replace=False)

for i in idx:
    df_working.loc[i, "salary"] = df_working["salary"].mean() * 10
    
    corruption_log.append({
        "row": int(i),
        "column": "salary",
        "issue": "outlier"
    })

In [78]:
idx = np.random.choice(df_working.index, size=int(0.1 * len(df_working)), replace=False)

for i in idx:
    df_working.loc[i, "email"] = "invalid_email"
    
    corruption_log.append({
        "row": int(i),
        "column": "email",
        "issue": "invalid_format"
    })

In [79]:
df_working.to_csv(os.path.join(SYNTHETIC_PATH, "corrupted_dataset.csv"), index=False)

In [80]:
corruption_log_df = pd.DataFrame(corruption_log)
corruption_log_df.to_csv(os.path.join(SYNTHETIC_PATH, "corruption_log.csv"), index=False)

In [81]:
# Load from disk (STRICT RULE)
df_check = pd.read_csv(os.path.join(SYNTHETIC_PATH, "corrupted_dataset.csv"))
log_check = pd.read_csv(os.path.join(SYNTHETIC_PATH, "corruption_log.csv"))

print("Shape of dataset:", df_check.shape)
print("Number of logged corruptions:", len(log_check))

# Basic validations
assert df_check.shape[0] >= 1000, "Dataset size mismatch"
assert len(log_check) > 0, "No corruption logged"

# Check presence of expected issues
issues_present = log_check["issue"].unique()

expected_issues = [
    "missing_value",
    "duplicate_row",
    "categorical_inconsistency",
    "whitespace_noise",
    "date_format_error",
    "outlier",
    "invalid_format"
]

for issue in expected_issues:
    assert issue in issues_present, f"Missing issue type: {issue}"

print("✅ All expected corruption types present")

# Check duplicates actually exist
duplicates_count = df_check.duplicated().sum()
print("Duplicate rows detected:", duplicates_count)
assert duplicates_count > 0, "No duplicates found"

print("✅ Dataset validation passed")

Shape of dataset: (1050, 7)
Number of logged corruptions: 1058
✅ All expected corruption types present
Duplicate rows detected: 15
✅ Dataset validation passed
